# 第六部分：数值计算 —— 从数学到代码的落地指南

## 第20章　浮点数陷阱与精度 —— 计算机不是数学

> 本章目标：深入理解 float32/float16/bfloat16 三种精度的物理边界——为什么 `0.1+0.2≠0.3`、为什么 `(1+1e-8)==1` 在 float32 下为 True、为什么梯度小于 1e-7 时参数停止更新。掌握混合精度训练的核心思想：前向/反向用 fp16 加速，权重副本用 fp32 保精度。
> 前置知识：第 1 章（浮点精度入门）、第 12 章（梯度下降）、第 19 章（损失函数）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")



### 20.1　二进制表示 —— 0.1 + 0.2 ≠ 0.3 的根本原因

第 1 章你第一次见到 `0.1 + 0.2 = 0.30000000000000004`。现在从二进制的底层重新审视这个问题——理解它不是为了炫技，而是为了理解为什么 float16 的最大值是 65504、为什么 bfloat16 能成为训练的"安全加速器"。

float32（IEEE 754 单精度）用 32 bits 表示一个数：
- 1 bit 符号位（sign）
- 8 bits 指数（exponent），范围 −126~127
- 23 bits 尾数（mantissa），约 7 位十进制有效数字

0.1 在二进制中是无限循环小数 `0.0001100110011...`，存入 23-bit 尾数时被截断——这就是误差的根源。**浮点数 = 有限精度下的有理近似，不是实数。**

📐 **定义　IEEE 754 单精度（float32）**：1 符号 + 8 指数 + 23 尾数。约 7 位十进制有效数字。最大正常值 ≈ 3.4×10³⁸，最小正正常值 ≈ 1.2×10⁻³⁸。

💻 **代码　亲手看 0.1 在 float32 下的真面目**




In [ ]:
import numpy as np

# float64 vs float32 下的 0.1
a64 = np.float64(0.1)
a32 = np.float32(0.1)

print(f"0.1 在 float64: {a64:.20f}")
print(f"0.1 在 float32: {a32:.20f}")
print(f"float64 误差: {a64 - 0.1:.2e}")
print(f"float32 误差: {a32 - 0.1:.2e}  ← float32 误差大得多!")

# 0.1 + 0.2 在不同精度下
print(f"\nfloat64: 0.1+0.2 = {np.float64(0.1)+np.float64(0.2):.20f}")
print(f"float32: 0.1+0.2 = {np.float32(0.1)+np.float32(0.2):.20f}")

# float32 的有效数字边界
print(f"\n1.0 + 1e-6 == 1.0 ? {np.float32(1.0) + np.float32(1e-6) == np.float32(1.0)}")  # False
print(f"1.0 + 1e-7 == 1.0 ? {np.float32(1.0) + np.float32(1e-7) == np.float32(1.0)}")  # True!
print(f"1.0 + 1e-8 == 1.0 ? {np.float32(1.0) + np.float32(1e-8) == np.float32(1.0)}")  # True!
print(f"结论: float32 只能区分约 1e-7 以上的变化")




> **关键洞察**：float32 的 7 位有效数字意味着**小于 1e-7 的相对变化对 float32 来说是看不见的。** 这直接导致第 20.4 节的"小梯度陷阱"——梯度值小于 1e-7 时，`param − lr × grad` 在 float32 下可能根本不产生任何变化。

---

### 20.2　float32 vs float64 —— 精度的代价

float64（双精度）用 64 bits：1 符号 + 11 指数 + 52 尾数，约 15-17 位有效数字。精度是 float32 的约 2000 万倍——但代价是 2 倍内存带宽和更慢的计算。

在深度学习中，大多数场景用 float32 作为**训练和推理的默认精度**——它提供了精度和效率的最佳平衡。float64 通常只在需要极高精度的场景使用（如科学计算、数值积分），而不会用于神经网络的常规训练。

💻 **代码　float32 vs float64 精度和内存全景对比**




In [ ]:
import numpy as np

# 精度对比: float32 能表示的最小正数 vs float64
print(f"float32 最小正正常数: {np.finfo(np.float32).tiny:.2e}")
print(f"float64 最小正正常数: {np.finfo(np.float64).tiny:.2e}")
print(f"float32 精度 (eps):    {np.finfo(np.float32).eps:.2e}")
print(f"float64 精度 (eps):    {np.finfo(np.float64).eps:.2e}")

# 内存对比
n = 10_000_000
arr32 = np.random.randn(n).astype(np.float32)
arr64 = np.random.randn(n).astype(np.float64)
print(f"\n{n/1e6:.0f}M 个元素:")
print(f"  float32: {arr32.nbytes / 1e6:.0f} MB")
print(f"  float64: {arr64.nbytes / 1e6:.0f} MB  ← 恰好是 float32 的 2 倍")

# 精度导致的"相等判断"陷阱
x32 = np.float32(1.0)
dx32 = np.float32(1e-7)
print(f"\nfloat32: 1.0 + 1e-7 == 1.0 ? {x32 + dx32 == x32}")   # True — 被吞了
x64 = np.float64(1.0)
dx64 = np.float64(1e-7)
print(f"float64: 1.0 + 1e-7 == 1.0 ? {x64 + dx64 == x64}")   # False — 精度够




---

### 20.3　float16 vs bfloat16 —— 训练加速的两种策略

AI 训练最大的瓶颈之一是**显存带宽**。float32 占用 4 bytes，如果能用 2 bytes 存储，显存直接省一半、计算吞吐翻倍。但 float16 的物理边界非常窄：

| 格式 | bits (符号/指数/尾数) | 最大正常值 | 最小正正常值 | 十进制精度 |
|------|----------------------|-----------|-------------|-----------|
| float32 | 1 / 8 / 23 | ~3.4×10³⁸ | ~1.2×10⁻³⁸ | ~7 位 |
| float16 | 1 / 5 / 10 | **65504** | ~6.1×10⁻⁵ | ~3 位 |
| bfloat16 | 1 / 8 / 7 | ~3.4×10³⁸ | ~1.2×10⁻³⁸ | ~2 位 |

**float16 的致命缺陷**：最大值只有 65504。训练中激活值或梯度稍微大一点就直接溢出变成 Inf。**bfloat16（Brain Floating Point）** 是深度学习的"天才发明"——指数位和 float32 一样（8 bits），所以**范围和 float32 一样大**，代价只是精度从 7 位降到 ~2 位。这对于神经网络来说完全可以接受——**参数的精确值不重要，重要的是梯度方向的相对大小。**

📐 **定义　混合精度训练（Mixed Precision Training）**：前向和反向传播用 fp16/bf16 加速，但维护一份 fp32 的权重副本。更新时在 fp32 副本上进行（保证小梯度不丢失），再将 fp32 权重截断回 fp16 用于下一轮前向。

💻 **代码　三种精度的数值边界 + softmax 鲁棒性对比**




In [ ]:
import numpy as np

# float16 的最大值硬边界
print(f"float16 最大值: {np.finfo(np.float16).max:.0f}")
print(f"  比它大 1 → 溢出为 inf: {np.float16(65505.0)}")

# 三种格式下 softmax([100, 0, -100]) 的表现
logits = np.array([100.0, 0.0, -100.0])

def softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

for dtype, name in [(np.float32, "float32"), (np.float16, "float16")]:
    try:
        x = np.array(logits, dtype=dtype)
        result = softmax(x.astype(np.float32))  # softmax 在 float32 下计算
        print(f"{name}: {result}")
    except Exception as e:
        print(f"{name}: ERROR — {str(e)[:50]}")

# bfloat16 模拟：float32 截断尾数到 7 bits
def to_bf16(x):
    """模拟 bfloat16: 保留 1+8=9 bits 的符号+指数, 尾数截断到 7 bits"""
    x32 = np.float32(x)
    # bf16 和 fp32 范围相同，只是精度更低
    # 实际推理/训练中由硬件（TPU/GPU tensor core）原生支持
    return x32  # 模拟——实际 bf16 由硬件处理

print(f"\nbf16 范围与 fp32 相同, {np.finfo(np.float32).max:.1e}")
print(f"bf16 不会溢出 → 训练比 fp16 安全得多")
print(f"混合精度 = fp16 提速 + fp32 保精度 → 工业标准")




---

### 20.4　小梯度陷阱 —— 当梯度小于 float32 的精度天花板

现在回到深度学习训练中最常见的数值陷阱。参数更新公式是 `θ -= lr × ∇L`。如果梯度值很小——比如 1e-8——而学习率是 1e-4，那么更新量 `lr × grad = 1e-12`。在 float32 下，`1.0 + 1e-12` 可能等于 `1.0`（因为 1e-12 < float32 的 eps ≈ 1.2e-7）。

这意味着：**梯度虽不为零，但更新量为零。参数停止更新，loss 停滞——不是因为到达最优，而是因为数值精度不够。**

这就是为什么训练后期 loss 看似收敛了，实际上可能只是"卡住"了。解决方案：(1) 增大学习率（不总是可行），(2) 用 float32 维护权重副本（混合精度），(3) 用 bfloat16 替代 float16（范围更大，不容易吞掉小梯度）。

💻 **代码　亲手验证小梯度时参数停止更新**




In [ ]:
import numpy as np

# float32 下的参数更新
param_f32 = np.float32(1.0)
lr = np.float32(1e-4)
grads = [np.float32(g) for g in [1e-3, 1e-5, 1e-6, 1e-7, 1e-8]]

print(f"初始参数: {param_f32}")
print(f"学习率: {lr}\n")
print(f"{'梯度':<12} {'更新量 lr×g':<15} {'新参数值':<15} {'实际变化?':<12}")
print("-" * 54)

for grad in grads:
    update = lr * grad
    new_param = param_f32 - update
    changed = new_param != param_f32
    print(f"{grad:<12.0e} {update:<15.0e} {new_param:<15.10f} "
          f"{'✓ 有更新' if changed else '✗ 无变化!'}")

print(f"\n结论: 梯度 ≤ 1e-7 时, lr×grad ≤ 1e-11")
print(f"      float32 的 eps ≈ 1.2e-7, 所以 1.0 − 1e-11 == 1.0")
print(f"      → 参数卡死,不是收敛,是数值精度耗尽了")




> **关键洞察**：`1.0 - 1e-11 == 1.0` 在 float32 下是 True。这不是 PyTorch 的 bug——是所有使用 IEEE 754 float32 的计算系统共享的物理边界。理解了这个，你就理解为什么混合精度训练必须维护 fp32 的权重副本——**fp16 的精度更差（~3 位），这个小梯度在 fp16 下早已被吞掉几百个 epoch 之前。**

🔗 **AI 连接**：第 24 章 Adam 优化器的 `eps=1e-8` 参数就是为了防止除以零——但同时也起到了"小梯度放大器"的作用（通过二阶矩归一化把小梯度放大）。第 31 章训练 GPT-2 时，`gradient_accumulation_steps` 让等效 batch size 变大 → 梯度变大 → 不容易被精度吞掉。

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）float16 的最大正常值是多少？为什么它容易溢出？bfloat16 如何解决了这个问题？
2. （概念）混合精度训练的核心思想是什么？为什么要维护 fp32 的权重副本？
3. （代码）用 `decimal` 模块精确计算 `0.1 + 0.2 == 0.3`，与 float32/float64 结果对比。
4. （代码）对 `logits=[100, 0, −100]`，分别用 float32、float16、以及"稳定版 softmax（先减 max）"计算 softmax。对比哪些会返回 NaN，哪些能正常输出。
---
> 预览：下一章——**归一化**，BatchNorm、LayerNorm、RMSNorm，把数值拉回安全区间。
